# Budget Library Testing Notebook

This notebook tests the budget data library functionality.


In [4]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from budgets import BudgetUpdater, Config, BudgetAPI, StorageManager


## 1. Test API Connection


In [5]:
api = BudgetAPI()

print("Available date range:")
print(f"  First: {api.get_first_date()[:7]}")
print(f"  Latest: {api.get_latest_date()[:7]}")

regions = api.get_regions()
print(f"\nTotal regions: {len(regions)}")
print("First 5 regions:")
for r in regions[:5]:
    print(f"  {r[0]}: {r[1]}")


Available date range:
  First: 2015-05
  Latest: 2025-09

Total regions: 90
First 5 regions:
  14000000: Белгородская область
  15000000: Брянская область
  17000000: Владимирская область
  20000000: Воронежская область
  24000000: Ивановская область


## 2. Test Storage Manager


In [6]:
storage = StorageManager()

# Load existing test data
income_old = storage.load_existing('../data/external/test_parquet.parquet')
expense_old = storage.load_existing('../data/external/test_expense.parquet')

print("Existing income data:")
print(f"  Shape: {income_old.shape}")
print(f"  Columns: {income_old.columns.tolist()}")
print(f"  Dates: {storage.get_existing_dates(income_old)}")

print("\nExisting expense data:")
print(f"  Shape: {expense_old.shape}")
print(f"  Columns: {expense_old.columns.tolist()}")


Existing income data:
  Shape: (385, 17)
  Columns: ['year', 'month', 'income_level', 'income_part', 'plan', 'adj_plan_consolidated', 'adj_plan_regional', 'adj_plan_growth_rate', 'execution_consolidated', 'execution_regional', 'growth_rate_regional', 'growth_rate_federal_district', 'growth_rate_russia', 'object_name', 'okato', 'oktmo', 'object_level']
  Dates: ['2015-05']

Existing expense data:
  Shape: (1386, 11)
  Columns: ['object_level', 'object_name', 'year', 'month', 'expense_level', 'expense_part', 'indicator_name', 'indicator_value', 'okato', 'oktmo', 'indicator_unit']


In [7]:
income_old.head()

,year,month,income_level,income_part,plan,adj_plan_consolidated,adj_plan_regional,adj_plan_growth_rate,execution_consolidated,execution_regional,growth_rate_regional,growth_rate_federal_district,growth_rate_russia,object_name,okato,oktmo,object_level
0,2015,5,0,ИТОГО ДОХОДОВ,65790667.0,73741433.47,57938214.0,0.950514,3.081041e+07,2.468728e+07,1.108331,1.112030,1.101295,Белгородская область,14000000,14000000,регион
1,2015,5,1,НАЛОГОВЫЕ И НЕНАЛОГОВЫЕ ДОХОДЫ,58709651.0,59496633.00,44019927.0,1.089682,2.305570e+07,1.697837e+07,1.096444,1.089834,1.117324,Белгородская область,14000000,14000000,регион
2,2015,5,2,"НАЛОГИ НА ПРИБЫЛЬ, ДОХОДЫ",34589166.0,34680621.00,27165057.0,1.101593,1.305610e+07,1.022939e+07,1.102889,1.092582,1.137538,Белгородская область,14000000,14000000,регион
3,2015,5,3,Налог на прибыль организаций,13688700.0,13688700.00,13688700.0,1.220393,5.411508e+06,5.411508e+06,1.278505,1.142339,1.245677,Белгородская область,14000000,14000000,регион
4,2015,5,4,"Налог на прибыль организаций, зачисляемый в бю...",13688700.0,13688700.00,13688700.0,1.220393,5.411507e+06,5.411507e+06,1.278505,1.142336,1.186578,Белгородская область,14000000,14000000,регион


## 3. Test Income Loader


In [8]:
from budgets import IncomeLoader

income_loader = IncomeLoader()

# Check what dates would be loaded
dates = income_loader.get_dates_to_load(date_from='2024-01', date_to='2024-01')
print(f"Dates to load: {dates}")


Loading from 2024-01 to 2024-01
Dates to load: [['2024-01-01T00:00:00.000Z']]


In [9]:
new_income = income_loader.download(date_from='2024-01', date_to='2024-01')
print(f"Downloaded {len(new_income)} rows")
new_income.head()


Loading from 2024-01 to 2024-01


Downloaded 59082 rows


,year,month,income_level,income_part,plan,adj_plan_consolidated,adj_plan_regional,adj_plan_growth_rate,execution_consolidated,execution_regional,growth_rate_regional,growth_rate_federal_district,growth_rate_russia,object_name,okato,oktmo,object_level
0,2024,1,0,ИТОГО ДОХОДОВ,163699997.6,163699997.6,132502361.6,0.826747,6.778726e+06,4.818946e+06,0.793455,1.330250,1.163585,Белгородская область,14000000,14000000,регион
1,2024,1,1,НАЛОГОВЫЕ И НЕНАЛОГОВЫЕ ДОХОДЫ,144757286.0,144757286.0,114233733.0,0.977558,6.528123e+06,4.417181e+06,1.430412,1.419546,1.373974,Белгородская область,14000000,14000000,регион
2,2024,1,2,"НАЛОГИ НА ПРИБЫЛЬ, ДОХОДЫ",98210253.0,98210253.0,78812240.0,0.992572,3.476379e+06,2.677641e+06,1.081425,1.537326,1.391359,Белгородская область,14000000,14000000,регион
3,2024,1,3,Налог на прибыль организаций,53484369.0,53484369.0,53484369.0,0.996141,1.484114e+06,1.484114e+06,1.570609,1.699505,1.574552,Белгородская область,14000000,14000000,регион
4,2024,1,4,"Налог на прибыль организаций, зачисляемый в бю...",42050647.0,42050647.0,42050647.0,0.956799,1.389783e+06,1.389783e+06,1.470780,1.651382,1.464617,Белгородская область,14000000,14000000,регион


## 4. Test Expense Loader


In [10]:
from budgets import ExpenseLoader

expense_loader = ExpenseLoader()

# Check what dates would be loaded
dates = expense_loader.get_dates_to_load(date_from='2024-01', date_to='2024-01')
print(f"Dates to load: {dates}")


Loading from 2024-01 to 2024-01
Dates to load: [['2024-01-01T00:00:00.000Z']]


In [11]:
new_expense = expense_loader.download(date_from='2024-01', date_to='2024-01')
print(f"Downloaded {len(new_expense)} rows")
new_expense.head()

Loading from 2024-01 to 2024-01


Downloaded 6737 rows


,year,month,expense_level,expense_part,plan,adj_plan_consolidated,adj_plan_regional,adj_plan_growth_rate,execution_consolidated,execution_regional,growth_rate_regional,growth_rate_federal_district,growth_rate_russia,object_name,okato,oktmo,object_level
0,2024,1,1,Расходы бюджета - ИТОГО,1.898704e+08,1.898704e+08,155314536.3,0.942025,6.635516e+06,5.245915e+06,0.830873,0.913242,1.034024,Белгородская область,14000000,14000000,регион
1,2024,1,1,ОБЩЕГОСУДАРСТВЕННЫЕ ВОПРОСЫ,1.240719e+07,1.240719e+07,7502234.6,1.625351,2.872108e+05,8.819724e+04,0.953798,1.187242,1.227252,Белгородская область,14000000,14000000,регион
2,2024,1,2,Функционирование высшего должностного лица суб...,4.367350e+04,4.367350e+04,10172.0,1.049128,1.478813e+03,1.541221e+02,1.525405,1.007503,1.217787,Белгородская область,14000000,14000000,регион
3,2024,1,2,Функционирование законодательных (представител...,3.465611e+05,3.465611e+05,247267.0,1.202812,6.924190e+03,1.827939e+03,0.917787,1.169260,1.151103,Белгородская область,14000000,14000000,регион
4,2024,1,2,Функционирование Правительства Российской Феде...,3.455248e+06,3.455248e+06,468395.6,1.008204,1.432514e+05,9.366083e+03,1.079599,1.131110,1.145612,Белгородская область,14000000,14000000,регион


## 5. Test Full Update Pipeline


In [15]:
updater = BudgetUpdater(data_dir='../data')

result = updater.update_income(
     date_from='2024-01',
     date_to='2024-01',
     input_file='../data/external/test_parquet.parquet',
     output_dir='../data/processed'
 )
print(result)

Loading from 2024-01 to 2024-01


Saved: ../data/processed/budget_income.parquet
Saved: ../data/processed/budget_income.csv
Saved: ../data/processed/budget_income.xlsx
{'status': 'updated', 'files': ['../data/processed/budget_income.parquet', '../data/processed/budget_income.csv', '../data/processed/budget_income.xlsx'], 'rows_added': 59082, 'total_rows': 59467}


## 6. Test Auto-detection of New Data


In [16]:
existing_dates = storage.get_existing_dates(income_old)
print(f"Existing dates in file: {existing_dates}")
print("The loader will automatically start from the next month after the last existing date")


Existing dates in file: ['2015-05']
The loader will automatically start from the next month after the last existing date


## 7. Test Save Formats


In [17]:
import os

test_df = income_old.head(100)
os.makedirs('../data/processed', exist_ok=True)

saved = storage.save_local(test_df, '../data/processed/test_output')
print(f"Saved files: {saved}")


Saved: ../data/processed/test_output.parquet
Saved: ../data/processed/test_output.csv
Saved: ../data/processed/test_output.xlsx
Saved files: ['../data/processed/test_output.parquet', '../data/processed/test_output.csv', '../data/processed/test_output.xlsx']


## CLI Usage Examples

```bash
# Update all data (auto-detect new months)
python -m budgets.main --type all --output-dir data/processed

# Update income only for specific date range
python -m budgets.main --type income --date-from 2024-01 --date-to 2024-03

# Update with S3 upload
python -m budgets.main --type all --s3-folder budgets --version v1.0.0
```
